In [1]:
class GaussianMixture(object):

    def __init__(self, k=3, max_iter=100, eps=1e-4, reg_cov=1e-6, random_state=None):
        self.k = k
        self.max_iter = max_iter
        self.eps = eps
        self.reg_cov = reg_cov
        self.random_state = random_state

        # Parameters to be estimated
        self.phi_ = None    # shape (k,)
        self.mu_ = None     # shape (k, n_features)
        self.sigma_ = None  # shape (k, n_features, n_features)

    def _log_gaussian(self, X, mu, sigma):
        """
        Compute log N(x | mu, sigma) for every row in X.
        """
        n_features = X.shape[1]
        diff = X - mu

        sign, log_det = np.linalg.slogdet(sigma)
        if sign <= 0:
            raise ValueError("Covariance matrix sigma must be positive definite.")

        # Solve sigma * z = diff.T instead of computing explicit inverse
        solved = np.linalg.solve(sigma, diff.T).T
        mahalanobis = np.sum(diff * solved, axis=1)

        return -0.5 * (n_features * np.log(2.0 * np.pi) + log_det + mahalanobis)

    @staticmethod
    def _logsumexp(A, axis=1):
        """Numerically stable log(sum(exp(A)))."""
        max_A = np.max(A, axis=axis, keepdims=True)
        result = max_A + np.log(np.sum(np.exp(A - max_A), axis=axis, keepdims=True))
        return np.squeeze(result, axis=axis)

    def _estimate_log_prob(self, X):
        """Log joint probability: log(phi_j * N(x | mu_j, sigma_j))."""
        log_prob = np.empty((X.shape[0], self.k))
        for j in range(self.k):
            log_prob[:, j] = np.log(self.phi_[j]) + self._log_gaussian(X, self.mu_[j], self.sigma_[j])
        return log_prob

    def _e_step(self, X):
        """
        E-step: calculate posterior responsibilities:
        w_j^(i) := p(z^(i) = j | x^(i); phi, mu, sigma)
        """
        log_joint = self._estimate_log_prob(X)
        log_norm = self._logsumexp(log_joint, axis=1)

        log_w = log_joint - log_norm[:, None]
        w = np.exp(log_w)  # w_j^(i)
        avg_log_likelihood = np.mean(log_norm)

        return w, avg_log_likelihood

    def _m_step(self, X, w):
        """
        M-step: update parameters phi, mu, and sigma based on weights w.
        """
        n_samples, n_features = X.shape

        sum_w = w.sum(axis=0) + 1e-10

        # phi_j := 1/m * sum_i w_j^(i)
        self.phi_ = sum_w / n_samples

        # mu_j := sum_i (w_j^(i) * x^(i)) / sum_i w_j^(i)
        self.mu_ = (w.T @ X) / sum_w[:, None]

        # sigma_j := sum_i w_j^(i) (x^(i) - mu_j)(x^(i) - mu_j)^T / sum_i w_j^(i)
        self.sigma_ = np.empty((self.k, n_features, n_features))
        for j in range(self.k):
            diff = X - self.mu_[j]
            cov = ((diff * w[:, j:j+1]).T @ diff) / sum_w[j]
            # Regularization to prevent singular covariance
            cov += self.reg_cov * np.eye(n_features)
            self.sigma_[j] = cov

    def fit(self, X):
        """
        Fit the Gaussian Mixture Model to data X via EM algorithm.
        """
        X = np.asarray(X, dtype=float)

        if X.ndim != 2 or X.shape[0] == 0:
            raise ValueError("X must be a non-empty 2D array.")
        if not np.all(np.isfinite(X)):
            raise ValueError("X must contain only finite values.")
        if not 1 <= self.k <= X.shape[0]:
            raise ValueError("k must be between 1 and n_samples.")

        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape

        # Initialize mu from randomly selected data points
        indices = rng.choice(n_samples, size=self.k, replace=False)
        self.mu_ = X[indices].copy()

        # Initialize uniform phi: phi_j = 1/k
        self.phi_ = np.full(self.k, 1.0 / self.k)

        # Initialize shared sigma from data covariance
        global_cov = np.atleast_2d(np.cov(X, rowvar=False)) + self.reg_cov * np.eye(n_features)
        self.sigma_ = np.repeat(global_cov[None, :, :], self.k, axis=0)

        self.lower_bound_ = -np.inf
        self.converged_ = False
        self.n_iter_ = 0
        self.log_likelihood_history_ = []

        # EM iterations
        for i in range(self.max_iter):
            # E-step
            w, log_likelihood = self._e_step(X)
            self.log_likelihood_history_.append(log_likelihood)

            # Check convergence
            if i > 0:
                improvement = log_likelihood - self.lower_bound_
                if abs(improvement) < self.eps:
                    self.converged_ = True
                    self.n_iter_ = i
                    break

            self.lower_bound_ = log_likelihood

            # M-step
            self._m_step(X, w)
            self.n_iter_ = i + 1

        # Final likelihood evaluation
        w, self.lower_bound_ = self._e_step(X)
        self.log_likelihood_history_.append(self.lower_bound_)

        return self

    def predict_proba(self, X):
        """Posterior probabilities w_j^(i) for each sample."""
        X = np.asarray(X, dtype=float)
        w, _ = self._e_step(X)
        return w

    def predict(self, X):
        """Return the most likely component (cluster assignment) for each sample."""
        return np.argmax(self.predict_proba(X), axis=1)

    def score_samples(self, X):
        """Return log p(x) for each sample."""
        X = np.asarray(X, dtype=float)
        return self._logsumexp(self._estimate_log_prob(X), axis=1)

    def score(self, X):
        """Return average log-likelihood across all samples."""
        return np.mean(self.score_samples(X))

In [2]:
import os
import h5py
import time
import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
from pandas import DataFrame
import matplotlib.pyplot as plt
from matplotlib import gridspec
%matplotlib inline

In [11]:
filenames = [
    "N-CMAPSS_DS01-005.h5",
    "N-CMAPSS_DS02-006.h5",
    "N-CMAPSS_DS03-012.h5",
    "N-CMAPSS_DS04.h5",
    "N-CMAPSS_DS05.h5",
    "N-CMAPSS_DS06.h5",
    "N-CMAPSS_DS07.h5",
    "N-CMAPSS_DS08a-009.h5",
    "N-CMAPSS_DS08c-008.h5"
]

In [12]:
### Set-up - Define file location
filename = "Data/N-CMAPSS_DS08a-009.h5"

In [13]:
# Time tracking, Operation time (min):  0.003
t = time.process_time()  

# Load data
with h5py.File(filename, 'r') as hdf:
        # Development set
        W_dev = np.array(hdf.get('W_dev'))             # W
        X_s_dev = np.array(hdf.get('X_s_dev'))         # X_s
        X_v_dev = np.array(hdf.get('X_v_dev'))         # X_v
        T_dev = np.array(hdf.get('T_dev'))             # T
        Y_dev = np.array(hdf.get('Y_dev'))             # RUL  
        A_dev = np.array(hdf.get('A_dev'))             # Auxiliary

        # Test set
        W_test = np.array(hdf.get('W_test'))           # W
        X_s_test = np.array(hdf.get('X_s_test'))       # X_s
        X_v_test = np.array(hdf.get('X_v_test'))       # X_v
        T_test = np.array(hdf.get('T_test'))           # T
        Y_test = np.array(hdf.get('Y_test'))           # RUL  
        A_test = np.array(hdf.get('A_test'))           # Auxiliary
        
        # Varnams
        W_var = np.array(hdf.get('W_var'))
        X_s_var = np.array(hdf.get('X_s_var'))  
        X_v_var = np.array(hdf.get('X_v_var')) 
        T_var = np.array(hdf.get('T_var'))
        A_var = np.array(hdf.get('A_var'))
        
        # from np.array to list dtype U4/U5
        W_var = list(np.array(W_var, dtype='U20'))
        X_s_var = list(np.array(X_s_var, dtype='U20'))  
        X_v_var = list(np.array(X_v_var, dtype='U20')) 
        T_var = list(np.array(T_var, dtype='U20'))
        A_var = list(np.array(A_var, dtype='U20'))
                          
W = np.concatenate((W_dev, W_test), axis=0)  
X_s = np.concatenate((X_s_dev, X_s_test), axis=0)
X_v = np.concatenate((X_v_dev, X_v_test), axis=0)
T = np.concatenate((T_dev, T_test), axis=0)
Y = np.concatenate((Y_dev, Y_test), axis=0) 
A = np.concatenate((A_dev, A_test), axis=0) 
    
print('')
print("Operation time (min): " , (time.process_time()-t)/60)
print('')
print ("W shape: " + str(W_dev.shape))
print ("X_s shape: " + str(X_s_dev.shape))
print ("X_v shape: " + str(X_v_dev.shape))
print ("T shape: " + str(T_dev.shape))
print ("A shape: " + str(A_dev.shape))


Operation time (min):  0.014285745550000003

W shape: (4885389, 4)
X_s shape: (4885389, 14)
X_v shape: (4885389, 14)
T shape: (4885389, 10)
A shape: (4885389, 4)


In [14]:
from sklearn.mixture import GaussianMixture


In [15]:
model = GaussianMixture(
    n_components=6,        # number of Gaussian components
    covariance_type='full',
    random_state=42
)

In [16]:
model.fit(X_s_dev)

,"n_components n_components: int, default=1The number of mixture components.",6
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to the method chosen to initialize theparameters (see `init_params`).In addition, it controls the generation of random samples from thefitted distribution (see the method `sample`).Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"covariance_type covariance_type: {'full', 'tied', 'diag', 'spherical'}, default='full'String describing the type of covariance parameters to use.Must be one of:- 'full': each component has its own general covariance matrix.- 'tied': all components share the same general covariance matrix.- 'diag': each component has its own diagonal covariance matrix.- 'spherical': each component has its own single variance.For an example of using `covariance_type`, refer to:ref:`sphx_glr_auto_examples_mixture_plot_gmm_selection.py`.",'full'
,"tol tol: float, default=1e-3The convergence threshold. EM iterations will stop when thelower bound average gain is below this threshold.",0.001
,"reg_covar reg_covar: float, default=1e-6Non-negative regularization added to the diagonal of covariance.Allows to assure that the covariance matrices are all positive.",1e-06
,"max_iter max_iter: int, default=100The number of EM iterations to perform.",100
,"n_init n_init: int, default=1The number of initializations to perform. The best results are kept.",1
,"init_params init_params: {'kmeans', 'k-means++', 'random', 'random_from_data'}, default='kmeans'The method used to initialize the weights, the means and theprecisions.String must be one of:- 'kmeans' : responsibilities are initialized using kmeans.- 'k-means++' : use the k-means++ method to initialize.- 'random' : responsibilities are initialized randomly.- 'random_from_data' : initial means are randomly selected data points... versionchanged:: v1.1 `init_params` now accepts 'random_from_data' and 'k-means++' as initialization methods.",'kmeans'
,"weights_init weights_init: array-like of shape (n_components, ), default=NoneThe user-provided initial weights.If it is None, weights are initialized using the `init_params` method.",None
,"means_init means_init: array-like of shape (n_components, n_features), default=NoneThe user-provided initial means,If it is None, means are initialized using the `init_params` method.",None
,"precisions_init precisions_init: array-like, default=NoneThe user-provided initial precisions (inverse of the covariancematrices).If it is None, precisions are initialized using the 'init_params'method.The shape depends on 'covariance_type':: (n_components,) if 'spherical', (n_features, n_features) if 'tied', (n_components, n_features) if 'diag', (n_components, n_features, n_features) if 'full'",None


In [18]:

import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# -------------------------------------------------------
# 1. Inspect RUL labels
# -------------------------------------------------------
y_dev = np.asarray(Y_dev).reshape(-1)
y_test = np.asarray(Y_test).reshape(-1)

assert len(X_s_dev) == len(y_dev)
assert len(X_s_test) == len(y_test)

print("Development RUL quantiles:")
print(np.quantile(y_dev, [0, .1, .25, .5, .75, .9, 1]))
print("Test RUL quantiles:")
print(np.quantile(y_test, [0, .1, .25, .5, .75, .9, 1]))

# -------------------------------------------------------
# 2. Define the detection objective
# -------------------------------------------------------
LOW_RUL = 20
HEALTHY_RUL = 80

y_true = (np.asarray(Y_test).reshape(-1) <= LOW_RUL).astype(int)
y_dev = np.asarray(Y_dev).reshape(-1)

healthy_idx = np.flatnonzero(y_dev > HEALTHY_RUL)

print("Total development observations:", len(y_dev))
print("Healthy observations:", len(healthy_idx))

if len(healthy_idx) < 2:
    raise ValueError(
        "Not enough healthy observations. "
        "Inspect the RUL distribution and adjust HEALTHY_RUL."
    )

# Shuffle healthy observations reproducibly.
rng = np.random.default_rng(42)
rng.shuffle(healthy_idx)

# Reserve 20% for threshold calibration.
n_cal = max(1, int(0.20 * len(healthy_idx)))

cal_idx = healthy_idx[:n_cal]
fit_candidates = healthy_idx[n_cal:]

# Limit training size AFTER reserving calibration observations.
n_fit = min(200_000, len(fit_candidates))
fit_idx = fit_candidates[:n_fit]

print("Healthy training observations:", len(fit_idx))
print("Healthy calibration observations:", len(cal_idx))

X_fit = X_s_dev[fit_idx]

# Fit the scaler using training observations only.
scaler = StandardScaler()
X_fit_scaled = scaler.fit_transform(X_fit)

# Calibration observations remain separate from model fitting.
X_cal_scaled = scaler.transform(X_s_dev[cal_idx])

# -------------------------------------------------------
# 3. Fit GMM
# -------------------------------------------------------
gmm = GaussianMixture(
    n_components=6,
    covariance_type="full",
    reg_covar=1e-5,
    max_iter=100,
    n_init=1,
    random_state=42,
)

gmm.fit(X_fit_scaled)

print("\nGMM converged:", gmm.converged_)
print("GMM iterations:", gmm.n_iter_)
print("GMM average training log-likelihood:", gmm.lower_bound_)

# -------------------------------------------------------
# 4. Fit Isolation Forest on the same training sample
# -------------------------------------------------------
iforest = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

iforest.fit(X_fit_scaled)

# -------------------------------------------------------
# 5. Score large datasets in batches
# -------------------------------------------------------
def score_in_batches(model, X, scaler, method, batch_size=20_000):
    scores = np.empty(len(X), dtype=np.float64)

    for start in range(0, len(X), batch_size):
        end = min(start + batch_size, len(X))
        X_batch = scaler.transform(X[start:end])

        if method == "gmm":
            # Higher means less likely under the GMM.
            scores[start:end] = -model.score_samples(X_batch)
        else:
            # Higher means more anomalous for Isolation Forest.
            scores[start:end] = -model.decision_function(X_batch)

    return scores


cal_gmm = score_in_batches(
    gmm, X_s_dev[cal_idx], scaler, "gmm"
)
cal_if = score_in_batches(
    iforest, X_s_dev[cal_idx], scaler, "iforest"
)

test_gmm = score_in_batches(
    gmm, X_s_test, scaler, "gmm"
)
test_if = score_in_batches(
    iforest, X_s_test, scaler, "iforest"
)

# Use the 95th percentile of healthy calibration scores
# as a fixed alarm threshold for each detector.
threshold_gmm = np.quantile(cal_gmm, 0.95)
threshold_if = np.quantile(cal_if, 0.95)

pred_gmm = (test_gmm >= threshold_gmm).astype(int)
pred_if = (test_if >= threshold_if).astype(int)

# -------------------------------------------------------
# 6. Evaluate both detectors
# -------------------------------------------------------
def evaluate(name, y_true, scores, predictions):
    print(f"\n===== {name} =====")
    print("ROC-AUC:", roc_auc_score(y_true, scores))
    print("Average Precision:", average_precision_score(y_true, scores))
    print("Precision:", precision_score(
        y_true, predictions, zero_division=0
    ))
    print("Recall:", recall_score(
        y_true, predictions, zero_division=0
    ))
    print("F1:", f1_score(
        y_true, predictions, zero_division=0
    ))
    print("Confusion matrix:\n", confusion_matrix(
        y_true, predictions
    ))

if np.unique(y_true).size < 2:
    raise ValueError(
        "The test set needs both low-RUL and non-low-RUL observations."
    )

print("\nTest low-RUL prevalence:", y_true.mean())

evaluate("GMM", y_true, test_gmm, pred_gmm)
evaluate("Isolation Forest", y_true, test_if, pred_if)


Development RUL quantiles:
[ 0.  6. 15. 32. 48. 60. 94.]
Test RUL quantiles:
[ 0.  6. 15. 30. 45. 54. 77.]
Total development observations: 4885389
Healthy observations: 58537
Healthy training observations: 46830
Healthy calibration observations: 11707

GMM converged: True
GMM iterations: 16
GMM average training log-likelihood: 43.79430290083837

Test low-RUL prevalence: 0.34499141417519275

===== GMM =====
ROC-AUC: 0.7397620123071449
Average Precision: 0.517586619102298
Precision: 0.43161918589353493
Recall: 0.996186552185375
F1: 0.6022853350216413
Confusion matrix:
 [[ 753671 1684924]
 [   4898 1279504]]

===== Isolation Forest =====
ROC-AUC: 0.5219733545426142
Average Precision: 0.3533518770604613
Precision: 0.36040962097462537
Recall: 0.565784699805824
F1: 0.44032714223173536
Confusion matrix:
 [[1148987 1289608]
 [ 557707  726695]]


In [19]:
print("RUL quantiles:")
print(np.quantile(Y_dev, [0, 0.1, 0.25, 0.5, 0.75, 0.9, 1]))

print("RUL <= 20:", np.sum(np.asarray(Y_dev).reshape(-1) <= 20))
print("RUL > 80:", np.sum(np.asarray(Y_dev).reshape(-1) > 80))

RUL quantiles:
[ 0.  6. 15. 32. 48. 60. 94.]
RUL <= 20: 1591836
RUL > 80: 58537


In [20]:

from sklearn.metrics import precision_recall_curve

# The GMM was fitted using fit_idx only.
# Select validation observations that were not used to fit it.
val_idx = np.setdiff1d(
    np.arange(len(X_s_dev)),
    fit_idx,
    assume_unique=False,
)

# Use a reproducible subset to limit memory and computation.
rng = np.random.default_rng(123)
n_val = min(100_000, len(val_idx))
val_idx = rng.choice(val_idx, size=n_val, replace=False)

y_val = (
    np.asarray(Y_dev).reshape(-1)[val_idx] <= LOW_RUL
).astype(int)

val_scores = score_in_batches(
    gmm, X_s_dev[val_idx], scaler, "gmm"
)

if np.unique(y_val).size < 2:
    raise ValueError("Validation subset must contain both classes.")

# Choose a threshold by maximizing validation F1.
precision, recall, thresholds = precision_recall_curve(
    y_val, val_scores
)

f1_values = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_values)
best_threshold = thresholds[best_idx]

print("Validation prevalence:", y_val.mean())
print("Selected threshold:", best_threshold)
print("Validation precision:", precision[best_idx])
print("Validation recall:", recall[best_idx])
print("Validation F1:", f1_values[best_idx])

# Apply the chosen threshold to the held-out test scores.
test_predictions = (test_gmm >= best_threshold).astype(int)

evaluate(
    "GMM with validation-selected threshold",
    y_true,
    test_gmm,
    test_predictions,
)


Validation prevalence: 0.32712
Selected threshold: -2.675491985374467
Validation precision: 0.5517816555465943
Validation recall: 0.920243335778919
Validation F1: 0.6898977861296133

===== GMM with validation-selected threshold =====
ROC-AUC: 0.7397620123071449
Average Precision: 0.517586619102298
Precision: 0.5096924932771728
Recall: 0.8911695870918918
F1: 0.6484898908220681
Confusion matrix:
 [[1337508 1101087]
 [ 139782 1144620]]


In [21]:
print("A_dev shape:", A_dev.shape)
print("A_test shape:", A_test.shape)

print("A_dev first 10 rows:")
print(A_dev[:10])

print("A_test first 10 rows:")
print(A_test[:10])

print("A_dev unique rows:", np.unique(A_dev, axis=0).shape[0])
print("A_test unique rows:", np.unique(A_test, axis=0).shape[0])

A_dev shape: (4885389, 4)
A_test shape: (3722997, 4)
A_dev first 10 rows:
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
A_test first 10 rows:
[[10.  1.  1.  1.]
 [10.  1.  1.  1.]
 [10.  1.  1.  1.]
 [10.  1.  1.  1.]
 [10.  1.  1.  1.]
 [10.  1.  1.  1.]
 [10.  1.  1.  1.]
 [10.  1.  1.  1.]
 [10.  1.  1.  1.]
 [10.  1.  1.  1.]]
A_dev unique rows: 611
A_test unique rows: 383
